# 01 · Data audit and cleaning

**Goal of this notebook:** : The audit looks for missing, duplicate and impossible data. It also summarises request types, sources and statuses, with the count and percentage of each. Some requests do not help answer our research questions, such as driver-filed reports and cancelled requests. We identify these with flags and exclude them from the main analysis.

In [ ]:
import duckdb, pandas as pd
from pathlib import Path
pd.set_option("display.max_columns", 50, "display.width", 140)
con = duckdb.connect()   # in-memory database; we read the CSVs directly with SQL

# Project root: notebooks run from notebooks/, but also work if launched from the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (ROOT / "data" / "raw").is_dir(), f"data/raw not found under {ROOT}"
RAW = str(ROOT / "data" / "raw" / "MyLA311_Service_Request_Data_202*.csv")
OUT = str(ROOT / "data" / "processed" / "requests_clean.parquet")
print(RAW)


## 1. Load the raw files with SQL
DuckDB reads all three yearly CSVs as one table without loading them into pandas memory.

In [2]:
con.execute(f"""
CREATE OR REPLACE VIEW raw AS
SELECT * FROM read_csv('{RAW}', header=true, all_varchar=true)
""")
con.sql("SELECT count(*) AS n_rows, count(DISTINCT SRNumber) AS n_unique_ids FROM raw").df()

,n_rows,n_unique_ids
0,4115521,4115521


**Check:** rows should equal unique request IDs. If not, there are duplicates to deal with.

## 2. Audit: what is in the data?

In [3]:
con.sql("""
SELECT Status, count(*) AS n, round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct
FROM raw GROUP BY 1 ORDER BY n DESC
""").df()

,Status,n,pct
0,Closed,3848352,93.51
1,Cancelled,158828,3.86
2,Pending,58390,1.42
3,Open,45052,1.09
4,Referred Out,4336,0.11
5,Forward,563,0.01


Statuses (per MyLA311 definitions): Closed = completed; Open and Pending = still in progress, so durations are censored and must be kept, not dropped; Forward = passed to another city department; Cancelled = could not be completed, so excluded from timing but possibly worth analysing as an outcome; Referred Out = handed to an outside agency and no longer trackable, so excluded from timing.

In [4]:
con.sql("""
SELECT RequestSource, count(*) AS n, round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct
FROM raw GROUP BY 1 ORDER BY n DESC
""").df()

,RequestSource,n,pct
0,Call,1612617,39.18
1,Mobile App,1329144,32.30
2,Self Service,657164,15.97
3,Driver Self Report,498834,12.12
4,Council's Office,9144,0.22
5,Email,7108,0.17
6,Voicemail,1043,0.03
7,Social,314,0.01
8,Queue Initiated Customer Call,47,0.00
9,Walk-in,47,0.00


In [5]:
con.sql("""
SELECT RequestType, count(*) AS n, round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct
FROM raw GROUP BY 1 ORDER BY n DESC
""").df()

,RequestType,n,pct
0,Bulky Items,1914546,46.52
1,Graffiti Removal,967960,23.52
2,Metal/Household Appliances,337878,8.21
3,Illegal Dumping Pickup,310022,7.53
4,Homeless Encampment,218592,5.31
5,Electronic Waste,116132,2.82
6,Dead Animal Removal,89623,2.18
7,Multiple Streetlight Issue,60980,1.48
8,Single Streetlight Issue,52662,1.28
9,Other,36316,0.88


In [6]:
con.sql("""
SELECT CreatedByUserOrganization, count(*) AS n
FROM raw GROUP BY 1 ORDER BY n DESC LIMIT 15
""").df()

,CreatedByUserOrganization,n
0,Self Service,1806824
1,LASAN,1342466
2,OCB,478332
3,ITA,267816
4,Self Service_SAN,173394
5,Council's Office,21638
6,Proactive Insert,21230
7,BSS,2223
8,LASAN Franchise,716
9,BSL,712


About 12% of requests are filed by city drivers, not residents. These reports bypass the resident’s decision to report, so they don’t reflect language barriers or any other reason a resident might not file. They would distort the reporting-gap analysis, because their locations follow truck routes. I exclude them from the main analysis and add them back in a sensitivity check to see whether the speed results change.

## 3. Parse dates and build a clean table
Dates arrive as text like `01/03/2023 12:43:39 PM`. We parse them, compute days-to-close, and **add flag columns instead of deleting rows** — so every exclusion is visible and reversible.

In [7]:
con.execute("""
CREATE OR REPLACE TABLE requests_clean AS
SELECT
    SRNumber                                              AS sr_number,
    strptime(CreatedDate, '%m/%d/%Y %I:%M:%S %p')         AS created_at,
    strptime(NULLIF(ClosedDate,''), '%m/%d/%Y %I:%M:%S %p') AS closed_at,
    RequestType                                           AS request_type,
    Status                                                AS status,
    RequestSource                                         AS request_source,
    TRY_CAST(NULLIF(Latitude,'')  AS DOUBLE)              AS lat,
    TRY_CAST(NULLIF(Longitude,'') AS DOUBLE)              AS lon,
    CD                                                    AS council_district,
    NCName                                                AS neighborhood_council
FROM raw
""")
con.execute("""
ALTER TABLE requests_clean ADD COLUMN days_to_close DOUBLE;
UPDATE requests_clean
SET days_to_close = date_diff('second', created_at, closed_at) / 86400.0
WHERE closed_at IS NOT NULL;
""")
con.sql("SELECT count(*) n, min(created_at) first_created, max(created_at) last_created, max(closed_at) last_closed FROM requests_clean").df()

,n,first_created,last_created,last_closed
0,4115521,2022-01-01 00:08:14,2024-12-31 23:59:45,2025-10-16 00:02:44


## 4. Cleaning rules (flags)
Each rule below is a column, so we can later run the analysis with and without it.

| Flag | Rule | Why |
|---|---|---|
| `bad_dates` | closed before it was created | impossible; a data entry error. Open requests are *not* flagged (NULL is turned into FALSE) |
| `no_coords` | latitude/longitude missing or 0 | cannot be assigned to a census tract |
| `is_driver_report` | source = `Driver Self Report` | filed by city staff, not residents |
| `is_cancelled` | status = `Cancelled` | "the request could not be completed", so there is no completion time. Exclude from timing; it may be worth analysing as an outcome |
| `is_referred_out` | status = `Referred Out` | "referred out to an outside agency and can no longer be tracked in MyLA311", so the outcome is unobservable. Exclude from timing |
| `still_open` | no close date | censored: still waiting at the data pull |

Status definitions are from the MyLA311 documentation: Open = being reviewed and worked on; Pending = reviewed, needs additional work; Forward = reviewed and forwarded to another department for fulfillment; Closed = completed. `Forward` is kept, because the request is still inside the city system.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE requests_clean AS
SELECT *,
    COALESCE(days_to_close < 0, FALSE)                    AS bad_dates,   -- COALESCE: open requests have no duration, so NULL must count as 'not bad'
    (lat IS NULL OR lon IS NULL OR lat = 0 OR lon = 0)    AS no_coords,
    (request_source = 'Driver Self Report')               AS is_driver_report,
    (status = 'Cancelled')                                AS is_cancelled,
    (status = 'Referred Out')                             AS is_referred_out,
    (closed_at IS NULL)                                   AS still_open
FROM requests_clean
""")
con.sql("""
SELECT sum(bad_dates::INT) bad_dates, sum(no_coords::INT) no_coords,
       sum(is_driver_report::INT) driver_reports, sum(is_cancelled::INT) cancelled,
       sum(is_referred_out::INT) referred_out, sum(still_open::INT) still_open
FROM requests_clean
""").df()

## 5. How long do requests take? (closed requests only — a first look)

In [ ]:
con.sql("""
SELECT request_type, count(*) n,
       round(median(days_to_close),2) median_days,
       round(quantile_cont(days_to_close,0.9),1) p90_days,
       round(quantile_cont(days_to_close,0.99),1) p99_days
FROM requests_clean
WHERE NOT bad_dates AND NOT is_cancelled AND NOT is_referred_out AND closed_at IS NOT NULL
GROUP BY 1 ORDER BY n DESC
""").df()

## 6. Save the clean table
Parquet is a compact, fast format; later notebooks read it instead of re-cleaning.

In [ ]:
con.execute(f"COPY requests_clean TO '{OUT}' (FORMAT parquet)")
con.sql(f"SELECT count(*) FROM '{OUT}'").df()